<a href="https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule in plain words

My baseline rule prioritizes content pages for **review** when they show signs of **staleness together with weak observed performance**.

I will use **content age (staleness)** as the main FlyRank flag-linked signal and combine it with a second observed performance signal. I will choose the exact thresholds only after checking the signal bucket tables, rather than setting arbitrary cutoffs in advance.

The rule will assign a higher `action_score` when both conditions suggest that a page may deserve attention. This score is only for **prioritization and decision-support**; it does not prove that the page is bad or that refreshing it will improve performance.

### Reason code

`STALE_WEAK_SIGNAL` — The content is relatively old and also shows a weak observed performance signal based on the thresholds supported by my signal checks.

### Action label

`REVIEW` — Review the content manually before deciding whether it should be refreshed, improved, kept unchanged, or given another action.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

file_path = "/content/drive/MyDrive/FlyRank Dataset/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

import pandas as pd
from pathlib import Path

queue = df.copy()

age_threshold = queue["content_age_days"].quantile(0.75)
engagement_threshold = queue["engagement_rate"].quantile(0.25)

print("Age threshold:", round(age_threshold, 2))
print("Engagement threshold:", round(engagement_threshold, 3))

queue["is_stale"] = queue["content_age_days"] >= age_threshold
queue["is_weak_engagement"] = queue["engagement_rate"] <= engagement_threshold

queue["action_score"] = (
    queue["is_stale"].astype(int) * 50
    + queue["is_weak_engagement"].astype(int) * 50
)

queue["reason_code"] = "NONE"

queue.loc[
    queue["is_stale"] & queue["is_weak_engagement"],
    "reason_code"
] = "STALE_WEAK_SIGNAL"

queue["action_label"] = "MONITOR"

queue.loc[
    queue["reason_code"] == "STALE_WEAK_SIGNAL",
    "action_label"
] = "REVIEW"

queue = queue.sort_values(
    by=["action_score", "content_age_days"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

output_columns = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "action_label",
    "content_age_days",
    "engagement_rate"
]

baseline_queue = queue[output_columns].copy()

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("\nCSV written to:")
print(output_path)

print("\nTotal ranked rows:", len(baseline_queue))

print("\nPages marked REVIEW:")
print((baseline_queue["action_label"] == "REVIEW").sum())

print("\nTop 10:")
display(baseline_queue.head(10))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Age threshold: 333.0
Engagement threshold: 0.0

CSV written to:
work/outputs/baseline_action_score.csv

Total ranked rows: 30000

Pages marked REVIEW:
5688

Top 10:


,rank,content_id,action_score,reason_code,action_label,content_age_days,engagement_rate
0,1,content_9d94766abf35,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
1,2,content_b6f1aab067b3,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
2,3,content_1da659104d64,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
3,4,content_d0ca9b3a8a26,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
4,5,content_d78761a6dc72,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
5,6,content_7424684ce198,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
6,7,content_ff832774fd69,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
7,8,content_4639346f504e,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
8,9,content_a0c3be8b0794,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0
9,10,content_4ff6831ce30d,100,STALE_WEAK_SIGNAL,REVIEW,557,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = top20["action_score"].apply(
    lambda x: "High - both rule signals triggered"
    if x == 100
    else "Medium - only one rule signal triggered"
    if x == 50
    else "Low - no rule signal triggered"
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda row:
    "The page may be intentionally evergreen, still useful to users, or its low engagement may have another explanation."
    if row["action_label"] == "REVIEW"
    else
    "Other important signals not included in this baseline rule may show that the page needs attention.",
    axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,content_id,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_9d94766abf35,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
1,2,content_b6f1aab067b3,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
2,3,content_1da659104d64,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
3,4,content_d0ca9b3a8a26,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
4,5,content_d78761a6dc72,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
5,6,content_7424684ce198,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
6,7,content_ff832774fd69,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
7,8,content_4639346f504e,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
8,9,content_a0c3be8b0794,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."
9,10,content_4ff6831ce30d,REVIEW,STALE_WEAK_SIGNAL,High - both rule signals triggered,"The page may be intentionally evergreen, still..."


## 3. Top-20 review

I manually reviewed the top 20 items produced by my baseline ranking rule.

For each item, I considered the recommended action, the reason code that placed it in the queue, my confidence in that recommendation, and circumstances that could make the recommendation wrong.

A high baseline score increases my confidence that the item matches the rule, but it does **not** prove that the recommended action is correct. For example, an older page with weak observed engagement may still be intentionally evergreen, useful to users, or affected by context that is not represented by my two-signal baseline.

Therefore, I treat this ranked queue as **decision-support for human review**, rather than an automatic decision about which content must be changed.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
rule_inputs = [
    "content_age_days",
    "engagement_rate"
]

blocked_inputs = [
    "trend_direction",
    "future_outcome",
    "future_window",
    "label",
    "product_flag",
    "refresh_flag",
    "quick_win_flag"
]

print("Rule inputs:")
for col in rule_inputs:
    print("-", col)

print("\nPotential leakage/product fields used in rule: NONE")

print("\nLeakage check:")
for col in blocked_inputs:
    if col in rule_inputs:
        print("WARNING:", col, "is used")
    else:
        print("PASS:", col, "not used")

Rule inputs:
- content_age_days
- engagement_rate

Potential leakage/product fields used in rule: NONE

Leakage check:
PASS: trend_direction not used
PASS: future_outcome not used
PASS: future_window not used
PASS: label not used
PASS: product_flag not used
PASS: refresh_flag not used
PASS: quick_win_flag not used


## 4. Weak picks + leakage check

### Weak picks

Some of the top-ranked `REVIEW` picks may be weak or incorrect recommendations because my baseline uses only two signals: `content_age_days` and `engagement_rate`.

For example, an old page with low engagement may still be useful, intentionally evergreen, or performing well on other signals that my rule does not consider. Similarly, low engagement alone does not prove that refreshing the content is the correct action.

Therefore, I treat these picks as candidates for **human review**, not automatic refresh decisions.

### Leakage check

I checked the inputs used by my baseline rule.

**Inputs used:**

* `content_age_days`
* `engagement_rate`

**Not used:**

* Future-window outcomes
* Label-derived columns
* `trend_direction` or other future outcome labels
* Existing FlyRank product/action flags
* Client identifiers

Therefore, the baseline score is based only on the two observed signals selected for my rule. No product flag or future-window/label-derived information is intentionally used in the score.

This keeps the baseline simple and honest so that a later model can be compared against it fairly.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.